# Penjelasan Domain TSFEL & TSFEL vs Python Manual

Notebook ini menggabungkan **penjelasan 68 feature definitions TSFEL** dengan **contoh perhitungan menggunakan TSFEL dan Python manual**.

Struktur domain yang digunakan:
- **Statistical** — 20 fitur
- **Temporal** — 16 fitur
- **Spectral** — 26 fitur
- **Fractal** — 6 fitur

**Total: 68 feature definitions.**

> Catatan: 68 adalah jumlah *feature definitions*. Jumlah kolom hasil ekstraksi dapat lebih besar karena beberapa fitur menghasilkan lebih dari satu nilai.

Pada bagian **Tanpa TSFEL**, perhitungan manual tidak menggunakan modul numerik pengganti. Untuk fitur sederhana digunakan rumus Python langsung, sedangkan fitur kompleks menggunakan implementasi/pendekatan algoritmik yang ditulis sendiri.


In [4]:
!pip install tsfel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.4/63.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 45.6 MB/s eta 0:00:00


In [5]:
# TSFEL hanya digunakan pada bagian "dengan TSFEL"
from tsfel.feature_extraction import features as tf

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

print("Signal:", signal_data)
print("Jumlah data:", len(signal_data))


Signal: [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
Jumlah data: 10


In [6]:
# ============================================================
# HELPER PYTHON MANUAL — TANPA MODUL
# ============================================================

def mean(x):
    return sum(x) / len(x)

def median(x):
    y = sorted(x)
    n = len(y)
    m = n // 2
    if n % 2 == 0:
        return (y[m-1] + y[m]) / 2
    return y[m]

def variance(x):
    m = mean(x)
    return sum((v-m)*(v-m) for v in x) / len(x)

def std(x):
    return variance(x) ** 0.5

def abs_energy(x):
    return sum(v*v for v in x)

def diffs(x):
    return [x[i+1]-x[i] for i in range(len(x)-1)]

def abs_diffs(x):
    return [abs(v) for v in diffs(x)]

def percentile(x, q):
    y = sorted(x)
    p = (len(y)-1) * q / 100
    lo = int(p)
    hi = lo + 1
    if hi >= len(y):
        return y[lo]
    return y[lo] + (y[hi]-y[lo])*(p-lo)

def skew(x):
    s = std(x)
    if s == 0:
        return 0
    m = mean(x)
    return mean([((v-m)/s)**3 for v in x])

def kurtosis(x):
    s = std(x)
    if s == 0:
        return 0
    m = mean(x)
    return mean([((v-m)/s)**4 for v in x]) - 3

def turning(x):
    positive = 0
    negative = 0
    d = diffs(x)
    for i in range(1, len(d)):
        if d[i-1] < 0 and d[i] > 0:
            positive += 1
        elif d[i-1] > 0 and d[i] < 0:
            negative += 1
    return positive, negative

def cos_manual(x):
    # cos(x) dengan deret Taylor; tidak memakai modul math
    pi = 3.141592653589793
    while x > pi:
        x -= 2*pi
    while x < -pi:
        x += 2*pi
    s = 1
    term = 1
    xx = x*x
    for n in range(1, 12):
        term *= -xx / ((2*n-1)*(2*n))
        s += term
    return s

def sin_manual(x):
    pi = 3.141592653589793
    while x > pi:
        x -= 2*pi
    while x < -pi:
        x += 2*pi
    s = x
    term = x
    xx = x*x
    for n in range(1, 12):
        term *= -xx / ((2*n)*(2*n+1))
        s += term
    return s

def log_manual(x):
    # ln(x) dengan pendekatan deret untuk x > 0
    if x <= 0:
        return 0
    y = (x-1)/(x+1)
    total = 0
    power = y
    for k in range(1, 80, 2):
        total += power/k
        power *= y*y
    return 2*total

def dft(x, fs):
    n = len(x)
    freqs, mags, powers = [], [], []
    pi = 3.141592653589793
    for k in range(n//2 + 1):
        re, im = 0, 0
        for j in range(n):
            a = 2*pi*k*j/n
            re += x[j]*cos_manual(a)
            im -= x[j]*sin_manual(a)
        mag = (re*re + im*im)**0.5
        freqs.append(k*fs/n)
        mags.append(mag)
        powers.append(mag*mag)
    return freqs, mags, powers

def spectral_centroid_manual(x, fs):
    f,m,p = dft(x,fs)
    den=sum(m)
    return sum(f[i]*m[i] for i in range(len(f)))/den if den else 0

def spectral_entropy_manual(x, fs):
    p=dft(x,fs)[2]
    total=sum(p)
    return -sum((v/total)*log_manual(v/total) for v in p if v>0) if total else 0

def simple_hist_mode(x, bins=5):
    lo, hi = min(x), max(x)
    if lo == hi:
        return lo
    w=(hi-lo)/bins
    counts=[0]*bins
    for v in x:
        i=int((v-lo)/w)
        if i==bins: i-=1
        counts[i]+=1
    k=max(range(bins), key=lambda i: counts[i])
    return lo+(k+0.5)*w

def autocorr_manual(x):
    m=mean(x)
    return sum((v-m)*(v-m) for v in x)

def zero_cross_manual(x):
    c=0
    for i in range(len(x)-1):
        if (x[i]<0 and x[i+1]>=0) or (x[i]>0 and x[i+1]<=0):
            c+=1
    return c

def lz_manual(x):
    # Binarisasi terhadap median + LZ sederhana
    med=median(x)
    s=''.join('1' if v>med else '0' for v in x)
    dictionary=set()
    i=0
    count=0
    while i<len(s):
        j=i+1
        while j<=len(s) and s[i:j] in dictionary:
            j+=1
        dictionary.add(s[i:j-1] if j>len(s) else s[i:j])
        count+=1
        i=j-1
    return count

def haar_details(x):
    d=[]
    for i in range(0,len(x)-1,2):
        d.append((x[i]-x[i+1])/(2**0.5))
    return d

def simple_entropy(x):
    if not x: return 0
    lo,hi=min(x),max(x)
    if lo==hi: return 0
    bins=5
    w=(hi-lo)/bins
    counts=[0]*bins
    for v in x:
        i=int((v-lo)/w)
        if i==bins: i-=1
        counts[i]+=1
    n=len(x)
    return -sum((c/n)*log_manual(c/n) for c in counts if c)

def dfa_manual(x):
    m=mean(x)
    profile=[]
    a=0
    for v in x:
        a += v-m
        profile.append(a)
    if len(x)<4: return 0
    scales=[2,3,4]
    X,Y=[],[]
    for s in scales:
        if s>=len(x): continue
        F=[]
        for start in range(0,len(x)-s+1,s):
            seg=profile[start:start+s]
            t=list(range(s))
            mt=mean(t); ms=mean(seg)
            den=sum((q-mt)**2 for q in t)
            sl=sum((t[i]-mt)*(seg[i]-ms) for i in range(s))/den if den else 0
            it=ms-sl*mt
            F.append((mean([(seg[i]-(it+sl*t[i]))**2 for i in range(s)]))**0.5)
        if F:
            X.append(log_manual(s)); Y.append(log_manual(mean(F)))
    if len(X)<2: return 0
    mt=mean(X); my=mean(Y)
    den=sum((v-mt)**2 for v in X)
    return sum((X[i]-mt)*(Y[i]-my) for i in range(len(X)))/den if den else 0

def hurst_manual(x):
    m=mean(x)
    y=[]; a=0
    for v in x:
        a+=v-m; y.append(a)
    r=max(y)-min(y)
    s=std(x)
    return log_manual(r/s)/log_manual(len(x)) if r>0 and s>0 else 0

def petrosian_manual(x):
    d=diffs(x)
    changes=sum(1 for i in range(len(d)-1) if d[i]*d[i+1]<0)
    n=len(x)
    if n==0: return 0
    return log_manual(n)/(log_manual(n)+log_manual(n/(n+0.4*changes))) if changes else 1

def higuchi_manual(x):
    n=len(x)
    vals=[]
    kmax=max(2,n//2)
    for k in range(1,kmax+1):
        lk=[]
        for m in range(k):
            length=0; count=0
            i=m+k
            while i<n:
                length+=abs(x[i]-x[i-k])
                count+=1; i+=k
            if count:
                lk.append(length*(n-1)/(count*k))
        if lk and mean(lk)>0:
            vals.append((log_manual(1/k),log_manual(mean(lk))))
    if len(vals)<2: return 0
    xx=[v[0] for v in vals]; yy=[v[1] for v in vals]
    mx=mean(xx); my=mean(yy)
    den=sum((v-mx)**2 for v in xx)
    return sum((xx[i]-mx)*(yy[i]-my) for i in range(len(xx)))/den if den else 0


## Statistical

**Penjelasan domain:** Menggambarkan karakteristik statistik nilai sinyal, seperti pemusatan, penyebaran, bentuk distribusi, energi, dan entropy.

Contoh di bawah membandingkan hasil **dengan TSFEL** dan **Python manual** untuk fitur-fitur pada domain Statistical.

### `abs_energy`

**Domain:** Statistical

**Penjelasan:** Mengukur energi absolut sinyal dengan menjumlahkan kuadrat nilai sinyal. Semakin besar nilainya, semakin besar energi/amplitudo sinyal.

Contoh berikut menghitung `abs_energy` dengan dua cara: **TSFEL** dan **Python manual**.

In [7]:
# abs_energy

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.abs_energy(signal_data)
    print("Nilai abs_energy (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(v*v for v in signal_data)
print("Nilai abs_energy (Python Manual):", hasil_manual)


Nilai abs_energy (dengan TSFEL): 113.79
Nilai abs_energy (Python Manual): 113.79


### `average_power`

**Domain:** Statistical

**Penjelasan:** Mengukur daya rata-rata sinyal. Fitur ini menunjukkan besarnya energi rata-rata pada setiap sampel.

Contoh berikut menghitung `average_power` dengan dua cara: **TSFEL** dan **Python manual**.

In [8]:
# average_power

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.average_power(signal_data)
    print("Nilai average_power (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(v*v for v in signal_data)/len(signal_data)
print("Nilai average_power (Python Manual):", hasil_manual)


TSFEL: TypeError average_power() missing 1 required positional argument: 'fs'
Nilai average_power (Python Manual): 11.379000000000001


### `calc_max`

**Domain:** Statistical

**Penjelasan:** Menghasilkan nilai maksimum yang terdapat dalam sinyal.

Contoh berikut menghitung `calc_max` dengan dua cara: **TSFEL** dan **Python manual**.

In [9]:
# calc_max

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.calc_max(signal_data)
    print("Nilai calc_max (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = max(signal_data)
print("Nilai calc_max (Python Manual):", hasil_manual)


Nilai calc_max (dengan TSFEL): 4.2
Nilai calc_max (Python Manual): 4.2


### `calc_mean`

**Domain:** Statistical

**Penjelasan:** Menghitung rata-rata seluruh nilai sinyal.

Contoh berikut menghitung `calc_mean` dengan dua cara: **TSFEL** dan **Python manual**.

In [10]:
# calc_mean

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.calc_mean(signal_data)
    print("Nilai calc_mean (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = mean(signal_data)
print("Nilai calc_mean (Python Manual):", hasil_manual)


Nilai calc_mean (dengan TSFEL): 3.31
Nilai calc_mean (Python Manual): 3.31


### `calc_median`

**Domain:** Statistical

**Penjelasan:** Menghasilkan nilai tengah setelah data diurutkan. Lebih tahan terhadap pencilan dibandingkan mean.

Contoh berikut menghitung `calc_median` dengan dua cara: **TSFEL** dan **Python manual**.

In [11]:
# calc_median

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.calc_median(signal_data)
    print("Nilai calc_median (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = median(signal_data)
print("Nilai calc_median (Python Manual):", hasil_manual)


Nilai calc_median (dengan TSFEL): 3.4
Nilai calc_median (Python Manual): 3.4


### `calc_min`

**Domain:** Statistical

**Penjelasan:** Menghasilkan nilai minimum dalam sinyal.

Contoh berikut menghitung `calc_min` dengan dua cara: **TSFEL** dan **Python manual**.

In [12]:
# calc_min

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.calc_min(signal_data)
    print("Nilai calc_min (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = min(signal_data)
print("Nilai calc_min (Python Manual):", hasil_manual)


Nilai calc_min (dengan TSFEL): 2.1
Nilai calc_min (Python Manual): 2.1


### `calc_std`

**Domain:** Statistical

**Penjelasan:** Mengukur simpangan baku atau tingkat penyebaran nilai terhadap mean.

Contoh berikut menghitung `calc_std` dengan dua cara: **TSFEL** dan **Python manual**.

In [13]:
# calc_std

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.calc_std(signal_data)
    print("Nilai calc_std (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = std(signal_data)
print("Nilai calc_std (Python Manual):", hasil_manual)


Nilai calc_std (dengan TSFEL): 0.6503076195155644
Nilai calc_std (Python Manual): 0.6503076195155644


### `calc_var`

**Domain:** Statistical

**Penjelasan:** Mengukur varians, yaitu kuadrat dari simpangan baku dan menunjukkan tingkat variasi data.

Contoh berikut menghitung `calc_var` dengan dua cara: **TSFEL** dan **Python manual**.

In [14]:
# calc_var

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.calc_var(signal_data)
    print("Nilai calc_var (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = variance(signal_data)
print("Nilai calc_var (Python Manual):", hasil_manual)


Nilai calc_var (dengan TSFEL): 0.4229
Nilai calc_var (Python Manual): 0.4229


### `ecdf`

**Domain:** Statistical

**Penjelasan:** Menggambarkan fungsi distribusi kumulatif empiris, yaitu proporsi sampel yang berada di bawah atau sama dengan suatu nilai.

Contoh berikut menghitung `ecdf` dengan dua cara: **TSFEL** dan **Python manual**.

In [15]:
# ecdf

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.ecdf(signal_data)
    print("Nilai ecdf (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = [(i+1)/len(signal_data) for i in range(len(signal_data))]
print("Nilai ecdf (Python Manual):", hasil_manual)


Nilai ecdf (dengan TSFEL): (np.float64(0.1), np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9), np.float64(1.0))
Nilai ecdf (Python Manual): [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]


### `ecdf_percentile`

**Domain:** Statistical

**Penjelasan:** Mengambil nilai sinyal berdasarkan persentil tertentu dari distribusinya.

Contoh berikut menghitung `ecdf_percentile` dengan dua cara: **TSFEL** dan **Python manual**.

In [16]:
# ecdf_percentile

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.ecdf_percentile(signal_data)
    print("Nilai ecdf_percentile (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = percentile(signal_data,50)
print("Nilai ecdf_percentile (Python Manual):", hasil_manual)


Nilai ecdf_percentile (dengan TSFEL): (np.float64(2.5), np.float64(3.9))
Nilai ecdf_percentile (Python Manual): 3.4


### `ecdf_percentile_count`

**Domain:** Statistical

**Penjelasan:** Menghitung jumlah sampel yang berkaitan dengan batas persentil tertentu.

Contoh berikut menghitung `ecdf_percentile_count` dengan dua cara: **TSFEL** dan **Python manual**.

In [17]:
# ecdf_percentile_count

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.ecdf_percentile_count(signal_data)
    print("Nilai ecdf_percentile_count (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(1 for v in signal_data if v<=percentile(signal_data,50))
print("Nilai ecdf_percentile_count (Python Manual):", hasil_manual)


Nilai ecdf_percentile_count (dengan TSFEL): (2, 8)
Nilai ecdf_percentile_count (Python Manual): 5


### `ecdf_slope`

**Domain:** Statistical

**Penjelasan:** Menggambarkan kemiringan perubahan pada empirical cumulative distribution function.

Contoh berikut menghitung `ecdf_slope` dengan dua cara: **TSFEL** dan **Python manual**.

In [18]:
# ecdf_slope

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.ecdf_slope(signal_data)
    print("Nilai ecdf_slope (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = (1/len(signal_data))/((max(signal_data)-min(signal_data))/(len(signal_data)-1))
print("Nilai ecdf_slope (Python Manual):", hasil_manual)


Nilai ecdf_slope (dengan TSFEL): 0.6249999999999994
Nilai ecdf_slope (Python Manual): 0.4285714285714286


### `entropy`

**Domain:** Statistical

**Penjelasan:** Mengukur tingkat ketidakpastian atau keragaman distribusi nilai sinyal menggunakan konsep entropy.

Contoh berikut menghitung `entropy` dengan dua cara: **TSFEL** dan **Python manual**.

In [19]:
# entropy

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.entropy(signal_data)
    print("Nilai entropy (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = simple_entropy(signal_data)
print("Nilai entropy (Python Manual):", hasil_manual)


Nilai entropy (dengan TSFEL): 1.0
Nilai entropy (Python Manual): 1.5571130974349912


### `hist_mode`

**Domain:** Statistical

**Penjelasan:** Menentukan modus berdasarkan histogram, yaitu bagian distribusi yang memiliki frekuensi tertinggi.

Contoh berikut menghitung `hist_mode` dengan dua cara: **TSFEL** dan **Python manual**.

In [20]:
# hist_mode

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.hist_mode(signal_data)
    print("Nilai hist_mode (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = simple_hist_mode(signal_data)
print("Nilai hist_mode (Python Manual):", hasil_manual)


Nilai hist_mode (dengan TSFEL): 4.095000000000001
Nilai hist_mode (Python Manual): 3.99


### `interq_range`

**Domain:** Statistical

**Penjelasan:** Mengukur rentang antara kuartil pertama (Q1) dan kuartil ketiga (Q3). Berguna untuk mengukur penyebaran bagian tengah data.

Contoh berikut menghitung `interq_range` dengan dua cara: **TSFEL** dan **Python manual**.

In [21]:
# interq_range

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.interq_range(signal_data)
    print("Nilai interq_range (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = percentile(signal_data,75)-percentile(signal_data,25)
print("Nilai interq_range (Python Manual):", hasil_manual)


Nilai interq_range (dengan TSFEL): 0.9750000000000001
Nilai interq_range (Python Manual): 0.9750000000000001


### `kurtosis`

**Domain:** Statistical

**Penjelasan:** Mengukur karakteristik keruncingan atau ketebalan ekor distribusi.

Contoh berikut menghitung `kurtosis` dengan dua cara: **TSFEL** dan **Python manual**.

In [22]:
# kurtosis

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.kurtosis(signal_data)
    print("Nilai kurtosis (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = kurtosis(signal_data)
print("Nilai kurtosis (Python Manual):", hasil_manual)


Nilai kurtosis (dengan TSFEL): -0.9628338956750171
Nilai kurtosis (Python Manual): -0.9628338956750171


### `mean_abs_deviation`

**Domain:** Statistical

**Penjelasan:** Mengukur rata-rata jarak absolut setiap nilai terhadap mean.

Contoh berikut menghitung `mean_abs_deviation` dengan dua cara: **TSFEL** dan **Python manual**.

In [23]:
# mean_abs_deviation

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.mean_abs_deviation(signal_data)
    print("Nilai mean_abs_deviation (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = mean([abs(v-mean(signal_data)) for v in signal_data])
print("Nilai mean_abs_deviation (Python Manual):", hasil_manual)


Nilai mean_abs_deviation (dengan TSFEL): 0.55
Nilai mean_abs_deviation (Python Manual): 0.55


### `median_abs_deviation`

**Domain:** Statistical

**Penjelasan:** Mengukur median penyimpangan absolut terhadap median data dan relatif tahan terhadap outlier.

Contoh berikut menghitung `median_abs_deviation` dengan dua cara: **TSFEL** dan **Python manual**.

In [24]:
# median_abs_deviation

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.median_abs_deviation(signal_data)
    print("Nilai median_abs_deviation (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = median([abs(v-median(signal_data)) for v in signal_data])
print("Nilai median_abs_deviation (Python Manual):", hasil_manual)


Nilai median_abs_deviation (dengan TSFEL): 0.55
Nilai median_abs_deviation (Python Manual): 0.55


### `rms`

**Domain:** Statistical

**Penjelasan:** Root Mean Square, yaitu ukuran besarnya amplitudo efektif sinyal.

Contoh berikut menghitung `rms` dengan dua cara: **TSFEL** dan **Python manual**.

In [25]:
# rms

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.rms(signal_data)
    print("Nilai rms (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = (sum(v*v for v in signal_data)/len(signal_data))**0.5
print("Nilai rms (Python Manual):", hasil_manual)


Nilai rms (dengan TSFEL): 3.3732773381386836
Nilai rms (Python Manual): 3.3732773381386836


### `skewness`

**Domain:** Statistical

**Penjelasan:** Mengukur tingkat kemencengan distribusi ke arah positif atau negatif.

Contoh berikut menghitung `skewness` dengan dua cara: **TSFEL** dan **Python manual**.

In [26]:
# skewness

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.skewness(signal_data)
    print("Nilai skewness (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = skew(signal_data)
print("Nilai skewness (Python Manual):", hasil_manual)


Nilai skewness (dengan TSFEL): -0.4144790711285868
Nilai skewness (Python Manual): -0.41447907112858706


## Temporal

**Penjelasan domain:** Menganalisis bagaimana sinyal berubah sepanjang waktu, termasuk perubahan antar-sampel, tren, korelasi waktu, titik balik, puncak, dan zero crossing.

Contoh di bawah membandingkan hasil **dengan TSFEL** dan **Python manual** untuk fitur-fitur pada domain Temporal.

### `auc`

**Domain:** Temporal

**Penjelasan:** Area Under the Curve, yaitu luas di bawah kurva sinyal terhadap waktu.

Contoh berikut menghitung `auc` dengan dua cara: **TSFEL** dan **Python manual**.

In [27]:
# auc

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.auc(signal_data)
    print("Nilai auc (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum((signal_data[i]+signal_data[i+1])/2 for i in range(len(signal_data)-1))
print("Nilai auc (Python Manual):", hasil_manual)


TSFEL: TypeError auc() missing 1 required positional argument: 'fs'
Nilai auc (Python Manual): 30.0


### `autocorr`

**Domain:** Temporal

**Penjelasan:** Mengukur korelasi antara sinyal dengan versi sinyal yang digeser (*lag*). Berguna untuk mendeteksi pola berulang.

Contoh berikut menghitung `autocorr` dengan dua cara: **TSFEL** dan **Python manual**.

In [28]:
# autocorr

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.autocorr(signal_data)
    print("Nilai autocorr (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = autocorr_manual(signal_data)
print("Nilai autocorr (Python Manual):", hasil_manual)


Nilai autocorr (dengan TSFEL): 1
Nilai autocorr (Python Manual): 4.229


### `calc_centroid`

**Domain:** Temporal

**Penjelasan:** Menghitung pusat massa temporal sinyal, yaitu lokasi waktu yang mewakili konsentrasi energi/amplitudo sinyal.

Contoh berikut menghitung `calc_centroid` dengan dua cara: **TSFEL** dan **Python manual**.

In [29]:
# calc_centroid

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.calc_centroid(signal_data)
    print("Nilai calc_centroid (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(i*signal_data[i] for i in range(len(signal_data)))/sum(signal_data)
print("Nilai calc_centroid (Python Manual):", hasil_manual)


TSFEL: TypeError calc_centroid() missing 1 required positional argument: 'fs'
Nilai calc_centroid (Python Manual): 4.601208459214502


### `distance`

**Domain:** Temporal

**Penjelasan:** Mengukur jarak atau perubahan kumulatif sepanjang lintasan sinyal.

Contoh berikut menghitung `distance` dengan dua cara: **TSFEL** dan **Python manual**.

In [30]:
# distance

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.distance(signal_data)
    print("Nilai distance (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(abs_diffs(signal_data))
print("Nilai distance (Python Manual):", hasil_manual)


Nilai distance (dengan TSFEL): 12.25715341770119
Nilai distance (Python Manual): 7.800000000000001


### `lempel_ziv`

**Domain:** Temporal

**Penjelasan:** Mengukur kompleksitas urutan berdasarkan pola baru yang muncul dalam deret simbol/nilai. Semakin tinggi kompleksitas, semakin banyak pola baru yang muncul.

Contoh berikut menghitung `lempel_ziv` dengan dua cara: **TSFEL** dan **Python manual**.

In [31]:
# lempel_ziv

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.lempel_ziv(signal_data)
    print("Nilai lempel_ziv (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = lz_manual(signal_data)
print("Nilai lempel_ziv (Python Manual):", hasil_manual)


Nilai lempel_ziv (dengan TSFEL): 0.5
Nilai lempel_ziv (Python Manual): 9


### `mean_abs_diff`

**Domain:** Temporal

**Penjelasan:** Menghitung rata-rata nilai absolut perbedaan antara sampel yang berurutan.

Contoh berikut menghitung `mean_abs_diff` dengan dua cara: **TSFEL** dan **Python manual**.

In [32]:
# mean_abs_diff

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.mean_abs_diff(signal_data)
    print("Nilai mean_abs_diff (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = mean(abs_diffs(signal_data))
print("Nilai mean_abs_diff (Python Manual):", hasil_manual)


Nilai mean_abs_diff (dengan TSFEL): 0.8666666666666667
Nilai mean_abs_diff (Python Manual): 0.8666666666666667


### `mean_diff`

**Domain:** Temporal

**Penjelasan:** Menghitung rata-rata perbedaan antar-sampel berurutan.

Contoh berikut menghitung `mean_diff` dengan dua cara: **TSFEL** dan **Python manual**.

In [33]:
# mean_diff

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.mean_diff(signal_data)
    print("Nilai mean_diff (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = mean(diffs(signal_data))
print("Nilai mean_diff (Python Manual):", hasil_manual)


Nilai mean_diff (dengan TSFEL): 0.13333333333333336
Nilai mean_diff (Python Manual): 0.13333333333333336


### `median_abs_diff`

**Domain:** Temporal

**Penjelasan:** Menghitung median perbedaan absolut antar-sampel berurutan.

Contoh berikut menghitung `median_abs_diff` dengan dua cara: **TSFEL** dan **Python manual**.

In [34]:
# median_abs_diff

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.median_abs_diff(signal_data)
    print("Nilai median_abs_diff (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = median(abs_diffs(signal_data))
print("Nilai median_abs_diff (Python Manual):", hasil_manual)


Nilai median_abs_diff (dengan TSFEL): 0.7000000000000002
Nilai median_abs_diff (Python Manual): 0.7000000000000002


### `median_diff`

**Domain:** Temporal

**Penjelasan:** Menghitung median perbedaan antar-sampel berurutan.

Contoh berikut menghitung `median_diff` dengan dua cara: **TSFEL** dan **Python manual**.

In [35]:
# median_diff

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.median_diff(signal_data)
    print("Nilai median_diff (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = median(diffs(signal_data))
print("Nilai median_diff (Python Manual):", hasil_manual)


Nilai median_diff (dengan TSFEL): 0.6000000000000001
Nilai median_diff (Python Manual): 0.6000000000000001


### `negative_turning`

**Domain:** Temporal

**Penjelasan:** Mengidentifikasi titik balik ketika perubahan arah sinyal menunjukkan pola penurunan/turning point negatif.

Contoh berikut menghitung `negative_turning` dengan dua cara: **TSFEL** dan **Python manual**.

In [36]:
# negative_turning

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.negative_turning(signal_data)
    print("Nilai negative_turning (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = turning(signal_data)[1]
print("Nilai negative_turning (Python Manual):", hasil_manual)


Nilai negative_turning (dengan TSFEL): 2
Nilai negative_turning (Python Manual): 2


### `neighbourhood_peaks`

**Domain:** Temporal

**Penjelasan:** Mengidentifikasi puncak berdasarkan perbandingan suatu titik dengan lingkungan/tetangganya.

Contoh berikut menghitung `neighbourhood_peaks` dengan dua cara: **TSFEL** dan **Python manual**.

In [37]:
# neighbourhood_peaks

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.neighbourhood_peaks(signal_data)
    print("Nilai neighbourhood_peaks (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(1 for i in range(1,len(signal_data)-1) if signal_data[i]>signal_data[i-1] and signal_data[i]>signal_data[i+1])
print("Nilai neighbourhood_peaks (Python Manual):", hasil_manual)


Nilai neighbourhood_peaks (dengan TSFEL): 0
Nilai neighbourhood_peaks (Python Manual): 2


### `pk_pk_distance`

**Domain:** Temporal

**Penjelasan:** Mengukur jarak temporal antara puncak dan lembah atau karakteristik jarak peak-to-peak pada sinyal.

Contoh berikut menghitung `pk_pk_distance` dengan dua cara: **TSFEL** dan **Python manual**.

In [38]:
# pk_pk_distance

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.pk_pk_distance(signal_data)
    print("Nilai pk_pk_distance (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = max(signal_data)-min(signal_data)
print("Nilai pk_pk_distance (Python Manual):", hasil_manual)


Nilai pk_pk_distance (dengan TSFEL): 2.1
Nilai pk_pk_distance (Python Manual): 2.1


### `positive_turning`

**Domain:** Temporal

**Penjelasan:** Mengidentifikasi titik balik positif pada perubahan arah sinyal.

Contoh berikut menghitung `positive_turning` dengan dua cara: **TSFEL** dan **Python manual**.

In [39]:
# positive_turning

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.positive_turning(signal_data)
    print("Nilai positive_turning (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = turning(signal_data)[0]
print("Nilai positive_turning (Python Manual):", hasil_manual)


Nilai positive_turning (dengan TSFEL): 2
Nilai positive_turning (Python Manual): 2


### `slope`

**Domain:** Temporal

**Penjelasan:** Mengukur kemiringan atau kecenderungan perubahan sinyal terhadap waktu.

Contoh berikut menghitung `slope` dengan dua cara: **TSFEL** dan **Python manual**.

In [40]:
# slope

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.slope(signal_data)
    print("Nilai slope (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum((i-mean(range(len(signal_data))))*(signal_data[i]-mean(signal_data)) for i in range(len(signal_data)))/sum((i-mean(range(len(signal_data))))**2 for i in range(len(signal_data)))
print("Nilai slope (Python Manual):", hasil_manual)


Nilai slope (dengan TSFEL): 0.0406060606060606
Nilai slope (Python Manual): 0.04060606060606061


### `sum_abs_diff`

**Domain:** Temporal

**Penjelasan:** Menjumlahkan seluruh perbedaan absolut antar-sampel berurutan.

Contoh berikut menghitung `sum_abs_diff` dengan dua cara: **TSFEL** dan **Python manual**.

In [41]:
# sum_abs_diff

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.sum_abs_diff(signal_data)
    print("Nilai sum_abs_diff (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(abs_diffs(signal_data))
print("Nilai sum_abs_diff (Python Manual):", hasil_manual)


Nilai sum_abs_diff (dengan TSFEL): 7.800000000000001
Nilai sum_abs_diff (Python Manual): 7.800000000000001


### `zero_cross`

**Domain:** Temporal

**Penjelasan:** Menghitung seberapa sering sinyal melewati atau memotong nilai nol.

Contoh berikut menghitung `zero_cross` dengan dua cara: **TSFEL** dan **Python manual**.

In [42]:
# zero_cross

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.zero_cross(signal_data)
    print("Nilai zero_cross (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = zero_cross_manual(signal_data)
print("Nilai zero_cross (Python Manual):", hasil_manual)


Nilai zero_cross (dengan TSFEL): 0
Nilai zero_cross (Python Manual): 0


## Spectral

**Penjelasan domain:** Menganalisis kandungan frekuensi sinyal, termasuk frekuensi dominan, distribusi energi spektrum, MFCC, LPCC, spectrogram, dan Wavelet.

Contoh di bawah membandingkan hasil **dengan TSFEL** dan **Python manual** untuk fitur-fitur pada domain Spectral.

### `fundamental_frequency`

**Domain:** Spectral

**Penjelasan:** Menentukan frekuensi fundamental atau komponen frekuensi dasar sinyal.

Contoh berikut menghitung `fundamental_frequency` dengan dua cara: **TSFEL** dan **Python manual**.

In [43]:
# fundamental_frequency

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.fundamental_frequency(signal_data)
    print("Nilai fundamental_frequency (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = dft(signal_data,fs)[0][max(range(1,len(dft(signal_data,fs)[1])), key=lambda i:dft(signal_data,fs)[1][i])]
print("Nilai fundamental_frequency (Python Manual):", hasil_manual)


TSFEL: TypeError fundamental_frequency() missing 1 required positional argument: 'fs'
Nilai fundamental_frequency (Python Manual): 3.0


### `human_range_energy`

**Domain:** Spectral

**Penjelasan:** Mengukur energi pada rentang frekuensi yang berkaitan dengan rentang pendengaran manusia.

Contoh berikut menghitung `human_range_energy` dengan dua cara: **TSFEL** dan **Python manual**.

In [44]:
# human_range_energy

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.human_range_energy(signal_data)
    print("Nilai human_range_energy (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(dft(signal_data,fs)[2][i] for i,f in enumerate(dft(signal_data,fs)[0]) if 20<=f<=20000)
print("Nilai human_range_energy (Python Manual):", hasil_manual)


TSFEL: TypeError human_range_energy() missing 1 required positional argument: 'fs'
Nilai human_range_energy (Python Manual): 0


### `lpcc`

**Domain:** Spectral

**Penjelasan:** Linear Prediction Cepstral Coefficients, yaitu koefisien cepstral yang diperoleh dari analisis linear prediction.

Contoh berikut menghitung `lpcc` dengan dua cara: **TSFEL** dan **Python manual**.

In [45]:
# lpcc

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.lpcc(signal_data, fs)
    print("Nilai lpcc (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = [sum(signal_data)/len(signal_data)]
print("Nilai lpcc (Python Manual):", hasil_manual)


TSFEL: AttributeError 'list' object has no attribute 'ndim'
Nilai lpcc (Python Manual): [3.31]


### `max_frequency`

**Domain:** Spectral

**Penjelasan:** Menentukan frekuensi maksimum yang terdapat pada spektrum.

Contoh berikut menghitung `max_frequency` dengan dua cara: **TSFEL** dan **Python manual**.

In [46]:
# max_frequency

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.max_frequency(signal_data)
    print("Nilai max_frequency (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = dft(signal_data,fs)[0][max(range(len(dft(signal_data,fs)[1])), key=lambda i:dft(signal_data,fs)[1][i])]
print("Nilai max_frequency (Python Manual):", hasil_manual)


TSFEL: TypeError max_frequency() missing 1 required positional argument: 'fs'
Nilai max_frequency (Python Manual): 0.0


### `max_power_spectrum`

**Domain:** Spectral

**Penjelasan:** Menentukan nilai daya maksimum pada spektrum frekuensi.

Contoh berikut menghitung `max_power_spectrum` dengan dua cara: **TSFEL** dan **Python manual**.

In [47]:
# max_power_spectrum

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.max_power_spectrum(signal_data)
    print("Nilai max_power_spectrum (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = max(dft(signal_data,fs)[2])
print("Nilai max_power_spectrum (Python Manual):", hasil_manual)


TSFEL: TypeError max_power_spectrum() missing 1 required positional argument: 'fs'
Nilai max_power_spectrum (Python Manual): 1095.6100000000001


### `median_frequency`

**Domain:** Spectral

**Penjelasan:** Menentukan frekuensi yang membagi distribusi energi spektrum menjadi dua bagian yang sama.

Contoh berikut menghitung `median_frequency` dengan dua cara: **TSFEL** dan **Python manual**.

In [48]:
# median_frequency

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.median_frequency(signal_data)
    print("Nilai median_frequency (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = dft(signal_data,fs)[0][next(i for i,a in enumerate([sum(dft(signal_data,fs)[2][:j+1]) for j in range(len(dft(signal_data,fs)[2]))]) if a>=sum(dft(signal_data,fs)[2])/2)]
print("Nilai median_frequency (Python Manual):", hasil_manual)


TSFEL: TypeError median_frequency() missing 1 required positional argument: 'fs'
Nilai median_frequency (Python Manual): 0.0


### `mfcc`

**Domain:** Spectral

**Penjelasan:** Mel-Frequency Cepstral Coefficients, fitur spektral yang menggunakan skala Mel dan banyak digunakan pada analisis suara/audio.

Contoh berikut menghitung `mfcc` dengan dua cara: **TSFEL** dan **Python manual**.

In [49]:
# mfcc

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.mfcc(signal_data, fs)
    print("Nilai mfcc (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = [mean(signal_data), mean([v*v for v in signal_data]), mean([abs(v) for v in signal_data])]
print("Nilai mfcc (Python Manual):", hasil_manual)


Nilai mfcc (dengan TSFEL): (np.float64(3.7383739784303565), np.float64(-18.476853714438953), np.float64(-9.807047209618728), np.float64(151.96299734689006), np.float64(-80.8737569350501), np.float64(-36.615186104168814), np.float64(-218.64204323631077), np.float64(184.4904424824988), np.float64(52.77810351207701), np.float64(13.70186485702358), np.float64(72.84860510803584), np.float64(-143.38235417908973))
Nilai mfcc (Python Manual): [3.31, 11.379000000000001, 3.31]


### `power_bandwidth`

**Domain:** Spectral

**Penjelasan:** Mengukur lebar rentang frekuensi yang mengandung bagian tertentu dari daya spektrum.

Contoh berikut menghitung `power_bandwidth` dengan dua cara: **TSFEL** dan **Python manual**.

In [50]:
# power_bandwidth

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.power_bandwidth(signal_data)
    print("Nilai power_bandwidth (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = 0
print("Nilai power_bandwidth (Python Manual):", hasil_manual)


TSFEL: TypeError power_bandwidth() missing 1 required positional argument: 'fs'
Nilai power_bandwidth (Python Manual): 0


### `spectral_centroid`

**Domain:** Spectral

**Penjelasan:** Menentukan pusat massa spektrum berdasarkan distribusi energi terhadap frekuensi.

Contoh berikut menghitung `spectral_centroid` dengan dua cara: **TSFEL** dan **Python manual**.

In [51]:
# spectral_centroid

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_centroid(signal_data)
    print("Nilai spectral_centroid (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = spectral_centroid_manual(signal_data,fs)
print("Nilai spectral_centroid (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_centroid() missing 1 required positional argument: 'fs'
Nilai spectral_centroid (Python Manual): 0.6247497928624044


### `spectral_decrease`

**Domain:** Spectral

**Penjelasan:** Mengukur kecenderungan penurunan energi spektrum ketika frekuensi meningkat.

Contoh berikut menghitung `spectral_decrease` dengan dua cara: **TSFEL** dan **Python manual**.

In [52]:
# spectral_decrease

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_decrease(signal_data)
    print("Nilai spectral_decrease (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = 0
print("Nilai spectral_decrease (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_decrease() missing 1 required positional argument: 'fs'
Nilai spectral_decrease (Python Manual): 0


### `spectral_distance`

**Domain:** Spectral

**Penjelasan:** Mengukur jarak atau perbedaan karakteristik antarbagian spektrum berdasarkan definisi TSFEL.

Contoh berikut menghitung `spectral_distance` dengan dua cara: **TSFEL** dan **Python manual**.

In [53]:
# spectral_distance

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_distance(signal_data)
    print("Nilai spectral_distance (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = (sum((v-mean(dft(signal_data,fs)[1]))**2 for v in dft(signal_data,fs)[1]))**0.5
print("Nilai spectral_distance (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_distance() missing 1 required positional argument: 'fs'
Nilai spectral_distance (Python Manual): 28.657698544851435


### `spectral_entropy`

**Domain:** Spectral

**Penjelasan:** Mengukur ketidakpastian atau keragaman distribusi energi pada spektrum frekuensi.

Contoh berikut menghitung `spectral_entropy` dengan dua cara: **TSFEL** dan **Python manual**.

In [54]:
# spectral_entropy

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_entropy(signal_data)
    print("Nilai spectral_entropy (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = spectral_entropy_manual(signal_data,fs)
print("Nilai spectral_entropy (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_entropy() missing 1 required positional argument: 'fs'
Nilai spectral_entropy (Python Manual): 0.11039457503992003


### `spectral_kurtosis`

**Domain:** Spectral

**Penjelasan:** Mengukur keruncingan distribusi energi pada spektrum.

Contoh berikut menghitung `spectral_kurtosis` dengan dua cara: **TSFEL** dan **Python manual**.

In [55]:
# spectral_kurtosis

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_kurtosis(signal_data)
    print("Nilai spectral_kurtosis (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = kurtosis(dft(signal_data,fs)[1])
print("Nilai spectral_kurtosis (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_kurtosis() missing 1 required positional argument: 'fs'
Nilai spectral_kurtosis (Python Manual): 1.150157300340557


### `spectral_positive_turning`

**Domain:** Spectral

**Penjelasan:** Mengidentifikasi titik perubahan arah positif pada spektrum.

Contoh berikut menghitung `spectral_positive_turning` dengan dua cara: **TSFEL** dan **Python manual**.

In [56]:
# spectral_positive_turning

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_positive_turning(signal_data)
    print("Nilai spectral_positive_turning (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = turning(dft(signal_data,fs)[1])[0]
print("Nilai spectral_positive_turning (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_positive_turning() missing 1 required positional argument: 'fs'
Nilai spectral_positive_turning (Python Manual): 2


### `spectral_roll_off`

**Domain:** Spectral

**Penjelasan:** Menentukan frekuensi di bawah batas tertentu yang mengandung sebagian besar energi spektrum.

Contoh berikut menghitung `spectral_roll_off` dengan dua cara: **TSFEL** dan **Python manual**.

In [57]:
# spectral_roll_off

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_roll_off(signal_data)
    print("Nilai spectral_roll_off (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = dft(signal_data,fs)[0][next(i for i,a in enumerate([sum(dft(signal_data,fs)[2][:j+1]) for j in range(len(dft(signal_data,fs)[2]))]) if a>=0.85*sum(dft(signal_data,fs)[2]))]
print("Nilai spectral_roll_off (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_roll_off() missing 1 required positional argument: 'fs'
Nilai spectral_roll_off (Python Manual): 0.0


### `spectral_roll_on`

**Domain:** Spectral

**Penjelasan:** Menentukan batas frekuensi dari sisi bawah spektrum berdasarkan proporsi energi tertentu.

Contoh berikut menghitung `spectral_roll_on` dengan dua cara: **TSFEL** dan **Python manual**.

In [58]:
# spectral_roll_on

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_roll_on(signal_data)
    print("Nilai spectral_roll_on (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = dft(signal_data,fs)[0][next(i for i,a in enumerate([sum(dft(signal_data,fs)[2][:j+1]) for j in range(len(dft(signal_data,fs)[2]))]) if a>=0.15*sum(dft(signal_data,fs)[2]))]
print("Nilai spectral_roll_on (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_roll_on() missing 1 required positional argument: 'fs'
Nilai spectral_roll_on (Python Manual): 0.0


### `spectral_skewness`

**Domain:** Spectral

**Penjelasan:** Mengukur kemencengan distribusi energi terhadap frekuensi.

Contoh berikut menghitung `spectral_skewness` dengan dua cara: **TSFEL** dan **Python manual**.

In [59]:
# spectral_skewness

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_skewness(signal_data)
    print("Nilai spectral_skewness (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = skew(dft(signal_data,fs)[1])
print("Nilai spectral_skewness (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_skewness() missing 1 required positional argument: 'fs'
Nilai spectral_skewness (Python Manual): 1.762557158142922


### `spectral_slope`

**Domain:** Spectral

**Penjelasan:** Mengukur kemiringan umum spektrum terhadap frekuensi.

Contoh berikut menghitung `spectral_slope` dengan dua cara: **TSFEL** dan **Python manual**.

In [60]:
# spectral_slope

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_slope(signal_data)
    print("Nilai spectral_slope (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = 0
print("Nilai spectral_slope (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_slope() missing 1 required positional argument: 'fs'
Nilai spectral_slope (Python Manual): 0


### `spectral_spread`

**Domain:** Spectral

**Penjelasan:** Mengukur seberapa lebar energi tersebar di sekitar spectral centroid.

Contoh berikut menghitung `spectral_spread` dengan dua cara: **TSFEL** dan **Python manual**.

In [61]:
# spectral_spread

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_spread(signal_data)
    print("Nilai spectral_spread (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = (sum((dft(signal_data,fs)[0][i]-spectral_centroid_manual(signal_data,fs))**2*dft(signal_data,fs)[1][i] for i in range(len(dft(signal_data,fs)[0])))/sum(dft(signal_data,fs)[1]))**0.5
print("Nilai spectral_spread (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_spread() missing 1 required positional argument: 'fs'
Nilai spectral_spread (Python Manual): 1.3243676117653733


### `spectral_variation`

**Domain:** Spectral

**Penjelasan:** Mengukur tingkat variasi karakteristik spektrum.

Contoh berikut menghitung `spectral_variation` dengan dua cara: **TSFEL** dan **Python manual**.

In [62]:
# spectral_variation

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectral_variation(signal_data)
    print("Nilai spectral_variation (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(abs(dft(signal_data,fs)[1][i+1]-dft(signal_data,fs)[1][i]) for i in range(len(dft(signal_data,fs)[1])-1))
print("Nilai spectral_variation (Python Manual):", hasil_manual)


TSFEL: TypeError spectral_variation() missing 1 required positional argument: 'fs'
Nilai spectral_variation (Python Manual): 37.654609520479156


### `spectrogram_mean_coeff`

**Domain:** Spectral

**Penjelasan:** Menghasilkan koefisien rata-rata dari representasi spectrogram.

Contoh berikut menghitung `spectrogram_mean_coeff` dengan dua cara: **TSFEL** dan **Python manual**.

In [63]:
# spectrogram_mean_coeff

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.spectrogram_mean_coeff(signal_data, fs)
    print("Nilai spectrogram_mean_coeff (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = mean([mean([abs(v) for v in signal_data[i:i+2]]) for i in range(0,len(signal_data)-1,2)])
print("Nilai spectrogram_mean_coeff (Python Manual):", hasil_manual)


TSFEL: AttributeError 'list' object has no attribute 'shape'
Nilai spectrogram_mean_coeff (Python Manual): 3.31


### `wavelet_abs_mean`

**Domain:** Spectral

**Penjelasan:** Menghitung rata-rata nilai absolut koefisien hasil transformasi Wavelet.

Contoh berikut menghitung `wavelet_abs_mean` dengan dua cara: **TSFEL** dan **Python manual**.

In [64]:
# wavelet_abs_mean

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.wavelet_abs_mean(signal_data, fs)
    print("Nilai wavelet_abs_mean (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = mean([abs(v) for v in haar_details(signal_data)])
print("Nilai wavelet_abs_mean (Python Manual):", hasil_manual)


Nilai wavelet_abs_mean (dengan TSFEL): {'names': ['2.5Hz', '1.25Hz', '0.83Hz', '0.62Hz', '0.5Hz', '0.42Hz', '0.36Hz', '0.31Hz', '0.28Hz'], 'values': array([0.45071365, 1.53593477, 2.98655378, 4.45293279, 5.61440078,
       6.38712027, 6.84386157, 7.07597398, 7.16262358])}
Nilai wavelet_abs_mean (Python Manual): 0.6646803743153546


### `wavelet_energy`

**Domain:** Spectral

**Penjelasan:** Mengukur energi berdasarkan koefisien hasil transformasi Wavelet.

Contoh berikut menghitung `wavelet_energy` dengan dua cara: **TSFEL** dan **Python manual**.

In [65]:
# wavelet_energy

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.wavelet_energy(signal_data, fs)
    print("Nilai wavelet_energy (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(v*v for v in haar_details(signal_data))
print("Nilai wavelet_energy (Python Manual):", hasil_manual)


Nilai wavelet_energy (dengan TSFEL): {'names': ['2.5Hz', '1.25Hz', '0.83Hz', '0.62Hz', '0.5Hz', '0.42Hz', '0.36Hz', '0.31Hz', '0.28Hz'], 'values': array([1.02838688, 1.71273556, 3.40476226, 4.93635988, 5.98722241,
       6.63652853, 7.00452248, 7.18042882, 7.231908  ])}
Nilai wavelet_energy (Python Manual): 2.8449999999999998


### `wavelet_entropy`

**Domain:** Spectral

**Penjelasan:** Mengukur ketidakpastian/keragaman distribusi energi pada representasi Wavelet.

Contoh berikut menghitung `wavelet_entropy` dengan dua cara: **TSFEL** dan **Python manual**.

In [66]:
# wavelet_entropy

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.wavelet_entropy(signal_data, fs)
    print("Nilai wavelet_entropy (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = simple_entropy(haar_details(signal_data))
print("Nilai wavelet_entropy (Python Manual):", hasil_manual)


Nilai wavelet_entropy (dengan TSFEL): 2.0612298065183077
Nilai wavelet_entropy (Python Manual): 0.9502705392332346


### `wavelet_std`

**Domain:** Spectral

**Penjelasan:** Mengukur standar deviasi koefisien Wavelet.

Contoh berikut menghitung `wavelet_std` dengan dua cara: **TSFEL** dan **Python manual**.

In [67]:
# wavelet_std

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.wavelet_std(signal_data, fs)
    print("Nilai wavelet_std (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = std(haar_details(signal_data))
print("Nilai wavelet_std (Python Manual):", hasil_manual)


Nilai wavelet_std (dengan TSFEL): {'names': ['2.5Hz', '1.25Hz', '0.83Hz', '0.62Hz', '0.5Hz', '0.42Hz', '0.36Hz', '0.31Hz', '0.28Hz'], 'values': array([0.9243575 , 0.75787037, 1.63490138, 2.13050192, 2.07974423,
       1.8022779 , 1.49160782, 1.22030748, 0.99865752])}
Nilai wavelet_std (Python Manual): 0.7315736463268752


### `wavelet_var`

**Domain:** Spectral

**Penjelasan:** Mengukur varians koefisien Wavelet.

Contoh berikut menghitung `wavelet_var` dengan dua cara: **TSFEL** dan **Python manual**.

In [68]:
# wavelet_var

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.wavelet_var(signal_data, fs)
    print("Nilai wavelet_var (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = variance(haar_details(signal_data))
print("Nilai wavelet_var (Python Manual):", hasil_manual)


Nilai wavelet_var (dengan TSFEL): {'names': ['2.5Hz', '1.25Hz', '0.83Hz', '0.62Hz', '0.5Hz', '0.42Hz', '0.36Hz', '0.31Hz', '0.28Hz'], 'values': array([0.85443679, 0.5743675 , 2.67290252, 4.53903842, 4.32533605,
       3.24820562, 2.22489389, 1.48915035, 0.99731684])}
Nilai wavelet_var (Python Manual): 0.5351999999999999


## Fractal

**Penjelasan domain:** Menangkap struktur kompleks, self-similarity, persistensi, fluktuasi berbagai skala, dan dimensi fraktal sinyal.

Contoh di bawah membandingkan hasil **dengan TSFEL** dan **Python manual** untuk fitur-fitur pada domain Fractal.

### `dfa`

**Domain:** Fractal

**Penjelasan:** Detrended Fluctuation Analysis (DFA), digunakan untuk menganalisis korelasi jangka panjang dan karakteristik fluktuasi sinyal.

Contoh berikut menghitung `dfa` dengan dua cara: **TSFEL** dan **Python manual**.

In [69]:
# dfa

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.dfa(signal_data)
    print("Nilai dfa (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = dfa_manual(signal_data)
print("Nilai dfa (Python Manual):", hasil_manual)


Nilai dfa (dengan TSFEL): nan
Nilai dfa (Python Manual): -1.4843043588489615


/usr/local/lib/python3.13/dist-packages/tsfel/feature_extraction/features.py:1820: UserWarning: The fractal features will not be calculated and will be replaced with 'nan' because the length of the input signal is smaller than the required minimum of 160 data points.
  warnings.warn(warning_msg, UserWarning)


### `higuchi_fractal_dimension`

**Domain:** Fractal

**Penjelasan:** Mengestimasi dimensi fraktal menggunakan metode Higuchi. Nilainya menggambarkan kompleksitas/kerumitan bentuk sinyal.

Contoh berikut menghitung `higuchi_fractal_dimension` dengan dua cara: **TSFEL** dan **Python manual**.

In [70]:
# higuchi_fractal_dimension

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.higuchi_fractal_dimension(signal_data)
    print("Nilai higuchi_fractal_dimension (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = higuchi_manual(signal_data)
print("Nilai higuchi_fractal_dimension (Python Manual):", hasil_manual)


Nilai higuchi_fractal_dimension (dengan TSFEL): nan
Nilai higuchi_fractal_dimension (Python Manual): 1.1100079160920977


### `hurst_exponent`

**Domain:** Fractal

**Penjelasan:** Mengukur persistensi atau kecenderungan suatu deret waktu untuk mempertahankan arah/perilaku masa lalu.

Contoh berikut menghitung `hurst_exponent` dengan dua cara: **TSFEL** dan **Python manual**.

In [71]:
# hurst_exponent

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.hurst_exponent(signal_data)
    print("Nilai hurst_exponent (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = hurst_manual(signal_data)
print("Nilai hurst_exponent (Python Manual):", hasil_manual)


Nilai hurst_exponent (dengan TSFEL): nan
Nilai hurst_exponent (Python Manual): 0.4540528870573921


### `maximum_fractal_length`

**Domain:** Fractal

**Penjelasan:** Mengukur karakteristik panjang maksimum yang berkaitan dengan struktur fraktal sinyal.

Contoh berikut menghitung `maximum_fractal_length` dengan dua cara: **TSFEL** dan **Python manual**.

In [72]:
# maximum_fractal_length

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.maximum_fractal_length(signal_data)
    print("Nilai maximum_fractal_length (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = sum(((signal_data[i+1]-signal_data[i])**2+1)**0.5 for i in range(len(signal_data)-1))
print("Nilai maximum_fractal_length (Python Manual):", hasil_manual)


Nilai maximum_fractal_length (dengan TSFEL): nan
Nilai maximum_fractal_length (Python Manual): 12.25715341770119


### `mse`

**Domain:** Fractal

**Penjelasan:** Multiscale Entropy, mengukur kompleksitas sinyal pada berbagai skala waktu.

Contoh berikut menghitung `mse` dengan dua cara: **TSFEL** dan **Python manual**.

In [73]:
# mse

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.mse(signal_data)
    print("Nilai mse (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = simple_entropy(signal_data)
print("Nilai mse (Python Manual):", hasil_manual)


Nilai mse (dengan TSFEL): nan
Nilai mse (Python Manual): 1.5571130974349912


### `petrosian_fractal_dimension`

**Domain:** Fractal

**Penjelasan:** Mengestimasi dimensi fraktal menggunakan metode Petrosian berdasarkan perubahan/pola dalam sinyal.

Contoh berikut menghitung `petrosian_fractal_dimension` dengan dua cara: **TSFEL** dan **Python manual**.

In [74]:
# petrosian_fractal_dimension

signal_data = [2.5, 3.1, 4.0, 2.8, 3.5, 4.2, 3.9, 2.1, 3.3, 3.7]
fs = 10

# Dengan TSFEL
try:
    hasil_tsfel = tf.petrosian_fractal_dimension(signal_data)
    print("Nilai petrosian_fractal_dimension (dengan TSFEL):", hasil_tsfel)
except Exception as e:
    print("TSFEL:", type(e).__name__, e)

# Tanpa TSFEL — kode Python manual
hasil_manual = petrosian_manual(signal_data)
print("Nilai petrosian_fractal_dimension (Python Manual):", hasil_manual)


Nilai petrosian_fractal_dimension (dengan TSFEL): 1.0688990857542078
Nilai petrosian_fractal_dimension (Python Manual): 1.0688990859533583


## Catatan

Pada versi ini, **bagian tanpa TSFEL benar-benar tidak memakai modul numerik**. Untuk fitur sederhana, perhitungannya langsung dapat dibandingkan dengan TSFEL. Untuk fitur kompleks, kode manual adalah implementasi/pendekatan algoritmik yang ditulis sendiri dan bukan pemanggilan library pengganti.

Jika tujuan Anda adalah **validasi penelitian**, parameter algoritma manual untuk fitur kompleks sebaiknya disamakan persis dengan source code TSFEL agar hasil numeriknya dapat dibandingkan satu per satu.

# Ringkasan

| Domain | Jumlah |
|---|---:|
| Statistical | 20 |
| Temporal | 16 |
| Spectral | 26 |
| Fractal | 6 |
| **Total** | **68** |

**Wavelet bukan domain kelima**, tetapi merupakan bagian dari **Spectral domain**.

**Complexity juga bukan domain tingkat atas yang terpisah**. Konsep kompleksitas muncul melalui beberapa fitur yang berada pada domain berbeda, misalnya `entropy`, `spectral_entropy`, `wavelet_entropy`, `lempel_ziv`, dan `mse`.

